___
# <center>A distribuição, a média e o risco</center>
___

## Aula 09

**Objetivo da aula:** ao final desta aula, você deve ser capaz de:

 * montar a distribuição de uma variável aleatória discreta numa tabela;
 * calcular esperança, variância e desvio padrão a partir dessa tabela;
 * conferir no código o que somar um fixo e o que multiplicar fazem com a média e com o risco;
 * decidir entre duas propostas usando os dois números, e não só a média.

Na lousa fizemos as contas do exemplo das audiências no braço. Aqui elas viram
três linhas de código, e é isso que libera espaço para a pergunta que importa:
o que muda quando a regra do jogo muda.


___
<div id="indice"></div>

## Índice

- [A distribuição numa tabela](#distribuicao)

- [Esperança](#esperanca)

- [Variância e desvio padrão](#variancia)

- [As duas propostas](#propostas)

- [O caso do acordo](#acordo)

- [De onde vem uma distribuição de verdade](#real)

- [RESUMO](#resumo)


___
<div id="distribuicao"></div>

# A distribuição numa tabela

O exemplo da aula: a advogada atende **duas audiências por dia**, cada acordo
rende **R$ 500**, e a chance de acordo em cada audiência é **0,20**.

A distribuição que montamos na lousa cabe num `DataFrame` de três linhas. A
coluna `x` traz os valores possíveis, e a coluna `p` a probabilidade de cada um.


In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

X = pd.DataFrame({
    "x": [0, 500, 1000],
    "p": [0.64, 0.32, 0.04],
})

X


A primeira conferência é sempre a mesma: **a coluna `p` soma 1?** Se não somar,
faltou um caminho da árvore ou sobrou um.


In [ ]:
X["p"].sum()


**✍️ Agora você.** De onde saiu o 0,32? Escreva a conta que produz a probabilidade de sair exatamente um acordo no dia, usando 0,20 e 0,80.


In [ ]:
# sai de dois caminhos: acordo na primeira e não na segunda, ou o contrário
0.20 * 0.80 + 0.80 * 0.20


[Volta ao Índice](#indice)


___
<div id="esperanca"></div>

# Esperança

$$E(X) = x_1 \, P(X = x_1) + x_2 \, P(X = x_2) + \cdots + x_k \, P(X = x_k)$$

Cada valor vezes a sua probabilidade, tudo somado. Em pandas isso é uma
multiplicação de colunas seguida de um `.sum()`.


In [ ]:
esperanca = (X["x"] * X["p"]).sum()

esperanca


R$ 200. Vale repetir o que a lousa disse: **200 não é um valor possível**. Num
dia ela recebe 0, 500 ou 1.000, nunca 200. É o que entra em média ao longo de
muitos dias.


**✍️ Agora você.** Na atividade em duplas, o lucro da semana valia 2.600 com probabilidade 0,16, 1.100 com 0,48 e −400 com 0,36. Monte a tabela e calcule a esperança. Confira que ela bate com os R$ 800 do quadro.


In [ ]:
L = pd.DataFrame({
    "x": [2600, 1100, -400],
    "p": [0.16, 0.48, 0.36],
})

print("soma das probabilidades:", L["p"].sum())
print("esperança:", (L["x"] * L["p"]).sum())


E a probabilidade de a semana dar prejuízo é a soma das probabilidades dos
valores negativos. Aqui só há um.


In [ ]:
L.loc[L["x"] < 0, "p"].sum()


[Volta ao Índice](#indice)


___
<div id="variancia"></div>

# Variância e desvio padrão

$$Var(X) = [x_1 - E(X)]^2 \, P(X = x_1) + \cdots + [x_k - E(X)]^2 \, P(X = x_k)$$

O desvio de cada valor até a esperança, ao quadrado, pesado pela probabilidade.


In [ ]:
variancia = (((X["x"] - esperanca) ** 2) * X["p"]).sum()

print("variância:", variancia)
print("desvio padrão:", variancia ** 0.5)


A variância sai em **reais ao quadrado**, que não quer dizer nada para ninguém.
O desvio padrão volta para reais, e é ele que se lê ao lado da esperança: um dia
típico fica em torno de R$ 200, oscilando algo como R$ 283 para cada lado.

Repare que a conta tem a mesma forma da esperança: cada linha da tabela entra
com o seu peso `p`. O que muda é o que está sendo pesado, o desvio até a média
em vez do próprio valor.


[Volta ao Índice](#indice)


___
<div id="propostas"></div>

# As duas propostas

O escritório vai mudar a remuneração:

- **proposta A**: um fixo de R$ 300 por dia, mais os honorários de sempre,
  ou seja $Y = X + 300$;
- **proposta B**: nenhum fixo, e os honorários triplicados, ou seja $W = 3X$.

Repare no que muda na tabela: **a coluna `p` é a mesma nas três**. O que a
transformação mexe é só na coluna `x`.


In [ ]:
def resumir(tabela, nome):
    """Esperança, variância e desvio padrão de uma distribuição em tabela."""
    E = (tabela["x"] * tabela["p"]).sum()
    V = (((tabela["x"] - E) ** 2) * tabela["p"]).sum()
    return {"quem": nome, "E": E, "Var": V, "DP": round(V ** 0.5, 2)}


Y = X.assign(x=X["x"] + 300)
W = X.assign(x=X["x"] * 3)

pd.DataFrame([
    resumir(X, "X: hoje"),
    resumir(Y, "Y = X + 300"),
    resumir(W, "W = 3X"),
])


Agora as propriedades da lousa, conferidas contra a tabela acima:

| propriedade | conta | resultado |
|---|---|---|
| $E(X + d) = E(X) + d$ | $200 + 300$ | 500 |
| $E(cX) = c \, E(X)$ | $3 \times 200$ | 600 |
| $Var(X + d) = Var(X)$ | 80.000 | 80.000 |
| $Var(cX) = c^2 \, Var(X)$ | $9 \times 80.000$ | 720.000 |

O fixo empurra a média e **não toca no risco**: somar 300 a todos os dias
desloca a distribuição inteira e não muda a distância de um dia para o outro.
Triplicar estica essas distâncias, e o quadrado da definição transforma o 3 em 9.


**✍️ Agora você.** B paga R$ 100 a mais por dia em média e oscila três vezes mais. Onde isso aparece: calcule, nas duas propostas, a probabilidade de o dia render MENOS de R$ 300.


In [ ]:
for tabela, nome in [(Y, "Y = X + 300"), (W, "W = 3X")]:
    print(nome, ":", tabela.loc[tabela["x"] < 300, "p"].sum())


Zero contra 0,64. O fixo da proposta A garante R$ 300 todo dia, e a B deixa a
advogada **sem nada em quase dois terços dos dias**, em troca de R$ 100 a mais
em média. Não há resposta certa: quem tem pouco caixa escolhe A, quem tem muito
escolhe B.


[Volta ao Índice](#indice)


___
<div id="acordo"></div>

# O caso do acordo

O caso que abriu a aula: 30% de chance de receber R$ 200.000, 70% de pagar
R$ 20.000, contra um acordo de R$ 40.000 na mesa.


In [ ]:
litigio = pd.DataFrame({
    "x": [200_000, -20_000],
    "p": [0.30, 0.70],
})

resumir(litigio, "ir a julgamento")


O julgamento vale R$ 46.000 em média, contra os R$ 40.000 do acordo. Pelo valor
esperado, ir a julgamento.

E o desvio padrão é de uns R$ 100.000, **mais que o dobro da própria média**. Em
70% dos cenários o cliente sai devendo. Aceitar menos que o valor esperado em
troca de certeza tem nome, **aversão ao risco**, e é o que explica a maior parte
dos acordos que fecham abaixo do valor esperado do julgamento.


**✍️ Agora você.** Qual o menor acordo que ainda empata com o julgamento no valor esperado? E se a chance de procedência caísse de 30% para 20%, quanto passaria a valer o julgamento?


In [ ]:
print("empata em:", (litigio["x"] * litigio["p"]).sum())

pessimista = pd.DataFrame({"x": [200_000, -20_000], "p": [0.20, 0.80]})
print("com 20% de chance:", (pessimista["x"] * pessimista["p"]).sum())


Com 20% de chance o julgamento passa a valer R$ 24.000, e o acordo de R$ 40.000
vira o melhor negócio até pelo valor esperado. **Dez pontos de probabilidade
viraram a decisão**, e é por isso que a avaliação de chance de êxito não é
detalhe de petição.


[Volta ao Índice](#indice)


___
<div id="real"></div>

# De onde vem uma distribuição de verdade

Até aqui as probabilidades vieram do enunciado. Na prática elas vêm de uma base:
a frequência com que cada valor apareceu é a estimativa da probabilidade dele.


In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

URL = "https://raw.githubusercontent.com/jtrecenti/202662-cdad2/main/dados"

criminal = pd.read_csv(f"{URL}/tjsp_cjsg_criminal.csv")

# Fora as linhas sem regime: não dá para calcular proporção de regime
# em acórdão que não informou regime nenhum.
penas = criminal.dropna(subset=["regime_inicial"])

penas.shape


In [ ]:
# a distribuição do regime inicial, lida direto da base
regime = (
    penas["regime_inicial"]
    .value_counts(normalize=True)
    .rename("p")
    .reset_index()
)

regime


Isso é uma distribuição: valores possíveis e a probabilidade de cada um, somando
1. A diferença é que aqui ela foi **estimada**, e não suposta.

Para calcular esperança precisamos de números, e regime é uma categoria. Então
vamos supor uma consequência: cada regime custa um valor diferente de honorários
de defesa.


**✍️ Agora você.** Suponha que a defesa cobre R$ 12.000 quando o regime é fechado, R$ 8.000 no semiaberto e R$ 5.000 no aberto. Qual o honorário esperado de um caso sorteado ao acaso na base?


In [ ]:
honorario = {"fechado": 12_000, "semiaberto": 8_000, "aberto": 5_000}

tabela = regime.assign(x=regime["regime_inicial"].map(honorario))

(tabela["x"] * tabela["p"]).sum()


É a mesma conta das audiências, com as probabilidades vindas dos dados em vez do
enunciado. Todo o resto da aula funciona igual.


[Volta ao Índice](#indice)


___
<div id="resumo"></div>

# RESUMO

1. **Distribuição** é a tabela inteira: valores possíveis e probabilidade de
   cada um. Conferir que `p` soma 1 é a primeira coisa a fazer.

2. **Esperança** é `(x * p).sum()`, e quase nunca é um valor possível.

3. **Variância** é `((x - E)**2 * p).sum()`, e o **desvio padrão** é a raiz dela,
   que é o número que se lê ao lado da esperança.

4. Somar um fixo mexe na média e **não** no risco. Multiplicar mexe nos dois, e
   no risco pelo quadrado.

5. Duas decisões com a mesma média podem ser muito diferentes. Quem só olha a
   média não vê a diferença.


[Volta ao Índice](#indice)
